# Full Guide

The purpose of this notebook is twofold: To serve as a comprehensive guide that demonstrates how to correctly use every feature of the tool and as a development tool in which every error and edge case is tested. As a result, the documentation in this notebook will lean on the verbose side. For a more concise and guided experience, refer to the official documentation.

In [218]:
import importlib
import setup_and_solve
importlib.reload(setup_and_solve)
from setup_and_solve import *

import warnings
warnings.filterwarnings('error')

## Constraint

The Constraint class forms the basic building block of a trajectory, and is best though of as a gate which the aircraft must hit. The minimum information required to define a constraint are the x, y, and z positions of the constraint relative to some origin (in progress: lat/lon/alt will also be supported). Optionally, a bearing constraint can be defined and enforced, as well as the tolerance with which the optimizer must hit the constaint. Shown below are two ways to create a default constraint with specified coordinates of (50, 0, 0).

In [219]:
con1 = Constraint(50, 0, 0)
con2 = Constraint(50, 0, 0, phi_deg=0.0, directional=False, tolerance=1.0)
display(con1)
display(con2)

Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0)

Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0)

The bearing angle is given in degrees and measures the angle from the positive x-axis in a counterclockwise fashion (in progress: azimuth will also be supported). The directional bool determines if the optimizer enforces the given bearing or not. The tolerance give the distance, in meters, that the optimizer is allowed to pass by the given point and consider the constraint met. Note that this cannot be set lower than 1 meter to prevent the problem from becoming too stiff.

In [220]:
try:
    Constraint(50, 0, 0, tolerance=0.5)
except ValueError as e:
    print(f"Error: {e}")

Error: tolerance must be >= 1.0.


Additionally, every Constraint has a property phi, which is the bearing in radians. This is not a settable value, and is used for the optimizer.

In [221]:
con3 = Constraint(50, 0, 0, phi_deg=180, directional=True)
display(con3.phi)

np.float64(3.141592653589793)

A more concise way to define a constraint is to use a vector and unpack it with *.

In [222]:
con4 = Constraint(*[100, 50, 0])
display(con4)

Constraint(x=100, y=50, z=0, phi_deg=0.0, directional=False, tolerance=1.0)

Every Constraint comes with the function state(), which returns the state of the constraint as a numpy vector. By default, this returns the three positional arguments in addition to the bearing in radians. Setting the argument full_size to False instead returns the positional arguments as a vector without the bearing.

In [223]:
display(con4.state())
display(con4.state(full_size=False))

array([100.,  50.,   0.,   0.])

array([100,  50,   0])

### StartConstraint

The StartConstraint class is a subclass of the Constraint class and is intended to define the starting state of the aircraft for the trajectory. A StartConstraint must include the three positional arguments in addition to the starting bearing. 

In [224]:
start_con1 = StartConstraint(0, 0, 0, 0)
display(start_con1)

StartConstraint(x=0, y=0, z=0, phi_deg=0, directional=True, tolerance=0.0)

Neither the directional nor the tolerance arguments can be set for a StartConstraint; they default to True and 0.0, respectively.

In [225]:
try:
    StartConstraint(0, 0, 0, 0, directional=True)
except ValueError as e:
    print(f"Error: {e}")

try:
    StartConstraint(0, 0, 0, 0, tolerance=0.0)
except ValueError as e:
    print(f"Error: {e}")

Error: Cannot set 'directional' for StartConstraint; defaults to True.
Error: Cannot set 'tolerance' for StartConstraint: defaults to 0.0.


### ExtendedConstraint

The ExtendedConstraint class is a subclass of the Constraint class intended to provide a more complex constraint than a simple gate. In addition to all the standard attributes of a Constraint, an ExtendedConstraint can take a radius [m], speed [m/s], and climb rate[m/s], as well as a tolerance for each which the optimizer will meet in the phase directly before the constraint (i.e., the optimizer will ensure that the trajectory immediately before the constraint has those controls). Lastly, an ExtendedConstraint has a duration attribute, which determines how long in seconds the optimizer will hold the given values for before the constraint (in progress: the optimizer will optionally hold the given values for the phase after the constraint, which can also be of any duration). Note that adding specific control targets that the optimizer must hit will likely require increasing the number of phases available per constraint to prevent creating an impossible trajectory. Shown below is a default ExtendedConstraint.

In [226]:
ex_con1 = ExtendedConstraint(50, 0, 0)
display(ex_con1)

ExtendedConstraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0, radius=None, radius_tol=None, speed=None, speed_tol=None, climb=None, climb_tol=None, duration=None)

Shown below is a fully defined ExtendedConstraint.

In [227]:
ex_con2 = ExtendedConstraint(200, 50, 0, 0, directional=False, tolerance=1.0, radius=30, radius_tol=0.0, speed=18, speed_tol=0.0, climb=4, climb_tol=0.0, duration=3)
display(ex_con2)

ExtendedConstraint(x=200, y=50, z=0, phi_deg=0, directional=False, tolerance=1.0, radius=30, radius_tol=0.0, speed=18, speed_tol=0.0, climb=4, climb_tol=0.0, duration=3)

Neither the duration nor the speed requested can be negative, and there is a warning if the duration is set too small to prevent the optimizer from effectively ignoring the constraint (i.e., meeting the constraint for a fraction of a second before the constraint, which gets ignored in the conversion to Ardupilot).

In [228]:
try:
    ExtendedConstraint(0, 0, 0, speed=-1)
except ValueError as e:
    print(f"Error: {e}")

try:
    ExtendedConstraint(0, 0, 0, duration=-1)
except ValueError as e:
    print(f"Error: {e}")

try:
    ExtendedConstraint(0, 0, 0, duration=0)
except ConstraintWarning as e:
    print(f"Warning: {e}")

Error: speed must be nonnegative.
Error: duration must be nonnegative.


Additionally, none of the tolerances can be negative. Note that the constraint tolerance is still required to be greater than or equal to 1.

In [229]:
try:
    ExtendedConstraint(0, 0, 0, radius=30, radius_tol=-1)
except ValueError as e:
    print(f"Error: {e}")

try:
    ExtendedConstraint(0, 0, 0, speed=18, speed_tol=-1)
except ValueError as e:
    print(f"Error: {e}")

try:
    ExtendedConstraint(0, 0, 0, climb=4, climb_tol=-1)
except ValueError as e:
    print(f"Error: {e}")

try:
    ExtendedConstraint(0, 0, 0, tolerance=0.0)
except ValueError as e:
    print(f"Error: {e}")

Error: radius_tol must be nonnegative.
Error: speed_tol must be nonnegative.
Error: climb_tol must be nonnegative.
Error: tolerance must be >= 1.0.


By default, all controls and corresponding tolerances along with duration are initialized as None. In order to activate an optional control requirement, the corresponding tolerance must be set. Setting the tolerance without specifying a target value will raise an error.

In [230]:
try:
    ExtendedConstraint(0, 0, 0, radius_tol=1.0)
except ConstraintError as e:
    print(f"Error: {e}")

try:
    ExtendedConstraint(0, 0, 0, speed_tol=1.0)
except ConstraintError as e:
    print(f"Error: {e}")

try:
    ExtendedConstraint(0, 0, 0, climb_tol=1.0)
except ConstraintError as e:
    print(f"Error: {e}")

Error: radius_tol is set, but radius is not.
Error: speed_tol is set, but speed is not.
Error: climb_tol is set, but climb is not.


Note that while preliminary checks are performed, checks against the airframe limits as given by Ardupilot are only carried out once an ExtendedConstraint (contained in a Trajectory, covered in the following section) is passed to the optimizer. There, some values may require changing based on the limits of the aircraft.

## Trajectory

The Trajectory class primarily serves as a container for a full trajectory, which is defined as a series of constraints which are met in sequential order. To initialize a Trajectory, a StartConstraint must be provided, as well as a list of Constraints. Below is a valid initialization of a Trajectory.

In [231]:
traj1 = Trajectory(start_con1, [con1, ex_con2])
display(traj1)

Trajectory(start=StartConstraint(x=0, y=0, z=0, phi_deg=0, directional=True, tolerance=0.0), constraints=[Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0), ExtendedConstraint(x=200, y=50, z=0, phi_deg=0, directional=False, tolerance=1.0, radius=30, radius_tol=0.0, speed=18, speed_tol=0.0, climb=4, climb_tol=0.0, duration=3)], phases_per_constraint=3)

Optionally, the number of optimizer phases used per constraint can be specified. By default, this is set to 3. Larger numbers may give better trajectories, but will also increase the runtime of the solution. Lower numbers risk impossible trajectories. To prevent this, all trajectories (even those with high allowed phase numbers) should be verified by the user to ensure flyability within the aircraft's parameters (in progress: automatic trajectory checking).

In [232]:
traj2 = Trajectory(start_con1, [con1], phases_per_constraint=3)
display(traj2)

Trajectory(start=StartConstraint(x=0, y=0, z=0, phi_deg=0, directional=True, tolerance=0.0), constraints=[Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0)], phases_per_constraint=3)

The Constraint passed to start must be of type StartConstraint, and the list of Constraints passed to constraints must contain at least one valid Constraint and cannot contain any StartConstraints.

In [233]:
try:
    Trajectory(con1, [con1])
except TypeError as e:
    print(f"Error: {e}")

try:
    Trajectory(start_con1, con1)
except TypeError as e:
    print(f"Error: {e}")

try:
    Trajectory(start_con1, [])
except ValueError as e:
    print(f"Error: {e}")

try:
    Trajectory(start_con1, [start_con1, con1])
except TypeError as e:
    print(f"Error: {e}")

Error: First Constraint must be a StartConstraint.
Error: Trajectory must be a list of Constraint objects.
Error: Trajectory must contain at least one constraint other than the StartConstraint.
Error: StartConstraints are not permitted in the constraint list.


The Trajectory class contains two methods which allow for the precise addition of Constraints to an existing Trajectory. One method allows for adding a constraint to the end of the trajectory, and the other allows for adding a constraint at a specified index. Note that the indexing includes the start constraint, so inserting a constraint at an index of 1 will place it between the start constraint and the first constraint (i.e., the inserted constraint will be the first element of the constraints list). Another way to think of this is that the constraint will be the index+1 constraint after the insertion (e.g., inserting a constraint at index 2 will put it after the second constraint, making it the third constraint). By default, the index is set to 1 and the constraint will be inserted between the start constraint and the first constraint.

In [234]:
traj2.add_constraint(con2)
display(traj2)

traj2.insert_constraint(con3, idx=2)
display(traj2)

Trajectory(start=StartConstraint(x=0, y=0, z=0, phi_deg=0, directional=True, tolerance=0.0), constraints=[Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0), Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0)], phases_per_constraint=3)

Trajectory(start=StartConstraint(x=0, y=0, z=0, phi_deg=0, directional=True, tolerance=0.0), constraints=[Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0), Constraint(x=50, y=0, z=0, phi_deg=180, directional=True, tolerance=1.0), Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0)], phases_per_constraint=3)

The Trajectory class contains two methods which perform the opposite functions of add_constraint() and insert_constraint() which are sub_constraint() and remove_constraint(), respectively.

In [235]:
traj2.add_constraint(con4)
display(traj2.trajectory())
traj2.sub_constraint()
display(traj2.trajectory())

traj2.insert_constraint(con4, 2)
display(traj2.trajectory())
traj2.remove_constraint(idx=2)
display(traj2.trajectory())

[StartConstraint(x=0, y=0, z=0, phi_deg=0, directional=True, tolerance=0.0),
 Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0),
 Constraint(x=50, y=0, z=0, phi_deg=180, directional=True, tolerance=1.0),
 Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0),
 Constraint(x=100, y=50, z=0, phi_deg=0.0, directional=False, tolerance=1.0)]

[StartConstraint(x=0, y=0, z=0, phi_deg=0, directional=True, tolerance=0.0),
 Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0),
 Constraint(x=50, y=0, z=0, phi_deg=180, directional=True, tolerance=1.0),
 Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0)]

[StartConstraint(x=0, y=0, z=0, phi_deg=0, directional=True, tolerance=0.0),
 Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0),
 Constraint(x=100, y=50, z=0, phi_deg=0.0, directional=False, tolerance=1.0),
 Constraint(x=50, y=0, z=0, phi_deg=180, directional=True, tolerance=1.0),
 Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0)]

[StartConstraint(x=0, y=0, z=0, phi_deg=0, directional=True, tolerance=0.0),
 Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0),
 Constraint(x=50, y=0, z=0, phi_deg=180, directional=True, tolerance=1.0),
 Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0)]

No constraint can be added before the StartConstraint, and StartConstraints are not allowed to be added or inserted into the list of constraints.

In [236]:
try:
    Trajectory(start_con1, [con1]).add_constraint(start_con1)
except TypeError as e:
    print(f"Error: {e}")

try:
    Trajectory(start_con1, [con1]).insert_constraint(start_con1)
except TypeError as e:
    print(f"Error: {e}")

try:
    Trajectory(start_con1, [con1]).insert_constraint(con2, 0)
except ValueError as e:
    print(f"Error: {e}")

Error: Can only add Constraint objects, not StartConstraint objects
Error: Can only insert Constraint objects, not StartConstraint objects
Error: Cannot insert constraint before start.


The start constraint can't be removed, and no constraint can be removed if there is only one constraint in the constraints list.

In [237]:
try:
    Trajectory(start_con1, [con1]).remove_constraint(idx=0)
except ValueError as e:
    print(f"Error: {e}")

try:
    Trajectory(start_con1, [con1]).sub_constraint()
except ValueError as e:
    print(f"Error: {e}")

try:
    Trajectory(start_con1, [con1]).remove_constraint()
except ValueError as e:
    print(f"Error: {e}")

Error: Cannot remove start constraint.
Error: Must have at least one constraint to define the trajectory.
Error: Must have at least one constraint to define the trajectory.


The Trajectory class also includes 2 helper methods which allow for easy extraction of the full trajectory. One returns the trajectory as a list of constraints (including the StartConstraint), and the other returns the trajectory as a numpy array containing the states (which are numpy arrays). For the state extraction, the argument full_size follows the same convention as the method of the same name for Constraints.

In [238]:
display(traj1.trajectory())
display(traj1.trajectory_states())
display(traj1.trajectory_states(full_size=False))

[StartConstraint(x=0, y=0, z=0, phi_deg=0, directional=True, tolerance=0.0),
 Constraint(x=50, y=0, z=0, phi_deg=0.0, directional=False, tolerance=1.0),
 ExtendedConstraint(x=200, y=50, z=0, phi_deg=0, directional=False, tolerance=1.0, radius=30, radius_tol=0.0, speed=18, speed_tol=0.0, climb=4, climb_tol=0.0, duration=3)]

array([[  0.,   0.,   0.,   0.],
       [ 50.,   0.,   0.,   0.],
       [200.,  50.,   0.,   0.]])

array([[  0,   0,   0],
       [ 50,   0,   0],
       [200,  50,   0]])

## ArdupilotParameters

The ArdupilotParameters class serves as a container for the relevant aircraft limits from Ardupilot, and provides an easy place to set and adjust these limits. One instance of ArdupilotParameters can be created for each aircraft, and the optimizer can easily understand the limits of the aircraft used. A default ArdupilotParameters object is explicitely defined below.

In [239]:
ap = ArdupilotParameters(
    V_max=30,
    V_min=10,
    V_cruise=22,
    max_climb = 5,
    max_desc=5,
    roll_limit=65,
    roll_min=5
)
display(ap)

ArdupilotParameters(V_max=30, V_min=10, V_cruise=22, max_climb=5, max_desc=5, roll_limit=65, roll_min=5)

The corresponding Ardupilot parameters are as follows:\
    V_max (float): AIRSPEED_MAX [m/s]\
    V_min (float): AIRSPEED_MIN [m/s]\
    V_cruise (float): AIRSPEED_CRUISE [m/s]\
    max_climb (float): TECS_CLIMB_MAX [m/s]\
    max_desc (float): TECS_SINK_MAX [m/s]\
    roll_limit (float): ROLL_LIMIT_DEG [deg]\
    roll_min (float): LEVEL_ROLL_LIMIT [deg]\
Note that the values are taken directly from the values given in the full parameter list. Modifying these values as necessary (e.g., making the sink rate negative) is handled by the optimizer. In addition to these parameters, the ArdupilotParameter class has 4 other properties: both roll limits in radians, and those roll limits converted to the turn radii in meters they correspond to.

In [240]:
display(ap.roll_min_rad)
display(ap.roll_limit_rad)
display(ap.min_turn)
display(ap.max_turn)

np.float64(0.08726646259971647)

np.float64(1.1344640137963142)

np.float64(23.0142715960108)

np.float64(564.1218269782739)

## OptimizerParameters

The OptimizerParameters class serves as a container for the parameters that control the optimizer. The first 4 weights control the objective function and determine which parameters the optimizer tries to minimize (and how hard it tries). The 4 weights are the time weight, turn weight, speed weight, and climb weight. Increasing the time weight will prioritize faster trajectories, increasing the turn weight will prioritize straighter trajectories, increasing the speed weight will prioritize keeping the speed close to the cruise speed, and increasing the climb weight will prioritize minimizing altitude changes and the rate of such changes. The segments and points attributes control the mesh used for the optimizer. Increasing these theoretically allows the optimizer to better represent the state of the aircraft and improve the optimization, though realistically this will only slow down the tool as the default values are more than good enough. The tol attribute controls the IPOPT tolerance used and is similar to the mesh controls in that decreasing it isn't necessary. Below is a default OptimizerParameters object defined explicitly.

In [241]:
optim_params = OptimizerParameters(
    time_weight=1,
    turn_weight=1,
    speed_weight=1,
    climb_weight=1,
    segments=10,
    points=10,
    tol=1e-8
)
display(optim_params)

OptimizerParameters(time_weight=1, turn_weight=1, speed_weight=1, climb_weight=1, segments=10, points=10, tol=1e-08)

The OptimizerParameters class also .has a method which returns just the weights as a numpy vector.

In [242]:
display(optim_params.get_weights())

array([1, 1, 1, 1])

## TrajectorySolution

The TrajectorySolution class is simply a container that the optimizer uses to store the results of the optimization process. It is not intended to be user definable, and examples of its use will be given in the Optimizer section.

## ConstraintWarning

The ConstraintWarning class extends the UserWarning class for easy user filtering and diagnosis.

## ConstraintError

The ConstraintError class extends the ValueError class for easy user filtering and diagnosis.

## Optimizer

The Optimizer class is the workhorse of the trajectory generation process, as it takes in a Trajectory, ArdupilotParameters, and OptimizerParameters and builds the problem, solves the problem, extracts the solution, and generates useful plots for visualization (in progress: exports solution to a file which can be directly imported into GCS for AP).

In [243]:
optim = Optimizer(traj1, ap, optim_params)

After initializing the Optimizer, it will check to see if there are any ExtendedConstraints so that the inupt values can be compared to the given Ardupilot limits. If the inputs fall outside of the Ardupilot parameters, then a warning will be thrown.

In [244]:
try:
    ex_con3 = ExtendedConstraint(0, 0, 0, radius=20, radius_tol=0)
    Optimizer(Trajectory(start_con1, [ex_con3]), ap, optim_params)
except ConstraintWarning as e:
    print(f"Warning: {e}")

try:
    ex_con3 = ExtendedConstraint(0, 0, 0, radius=-600, radius_tol=0)
    Optimizer(Trajectory(start_con1, [ex_con3]), ap, optim_params)
except ConstraintWarning as e:
    print(f"Warning: {e}")

try:
    ex_con3 = ExtendedConstraint(0, 0, 0, speed=9, speed_tol=0)
    Optimizer(Trajectory(start_con1, [ex_con3]), ap, optim_params)
except ConstraintWarning as e:
    print(f"Warning: {e}")

try:
    ex_con3 = ExtendedConstraint(0, 0, 0, speed=31, speed_tol=0)
    Optimizer(Trajectory(start_con1, [ex_con3]), ap, optim_params)
except ConstraintWarning as e:
    print(f"Warning: {e}")

try:
    ex_con3 = ExtendedConstraint(0, 0, 0, climb=6, climb_tol=0)
    Optimizer(Trajectory(start_con1, [ex_con3]), ap, optim_params)
except ConstraintWarning as e:
    print(f"Warning: {e}")

try:
    ex_con3 = ExtendedConstraint(0, 0, 0, climb=-6, climb_tol=0)
    Optimizer(Trajectory(start_con1, [ex_con3]), ap, optim_params)
except ConstraintWarning as e:
    print(f"Warning: {e}")

Broadly speaking, the solution pathway has 4 sequential steps, which are:\
setup() -> solve() -> extract() -> plot()/export()\
The first method, setup(), takes the Trajectory, the ArdupilotParameters, and the OptimizerParameters and forms the YAPSS problem to be solved. Calling solve() without first calling setup() will automatically run setup(), but it's better practice to manually call setup() before solve() if only to reduce the potential overhead. Calling this method will return the YAPSS problem object.

In [247]:
optim.setup()

After calling this method, the problem is stored in the Optimizer.

In [248]:
display(optim.problem)

The solve() method runs YAPSS to solve the optimization problem (which calls IPOPT under the hood) and returns the YAPSS solution object.

In [250]:
optim.solve()

<yapss._private.solution.Solution: 'Constraints to Trajectory'>

Once solve() is called, the solution object is stored in the Optimizer.

In [251]:
display(optim.solution)

<yapss._private.solution.Solution: 'Constraints to Trajectory'>